In [ ]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Created on Mon Feb 22 20:09:58 2021

@author: ronguy
"""

import numpy as np
#from sklearn.manifold import TSNE
from scipy import integrate as int
from scipy import stats
import matplotlib.pyplot as plt
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from time import time
from sklearn.preprocessing import StandardScaler
from sklearn import preprocessing
from sklearn import datasets
from sklearn.neural_network import MLPRegressor
from sklearn.linear_model import LinearRegression
import statsmodels.stats.weightstats as ws
from sklearn.cluster import KMeans
import umap
from lmfit import minimize, Parameters
import matplotlib 
from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest


from sklearn.cluster import DBSCAN
from sklearn import metrics
from sklearn.datasets import make_blobs
from sklearn.preprocessing import StandardScaler
from sklearn.neural_network import MLPRegressor
from sklearn.metrics import accuracy_score
from sklearn.preprocessing import MinMaxScaler


import matplotlib.pyplot as pl
from sklearn.cluster import AgglomerativeClustering

# Define a bunch of functions

In [ ]:
def TestHetero(DB1,DB2,UMAPMRK,CNum=20000,fname=None):
        from scipy.spatial import distance
        CAll=pd.concat([
                DB1.sample(CNum,random_state=42),
                DB2.sample(CNum,random_state=42)
        ]).copy()
        X_2d=draw_umap(CAll[UMAPMRK],cc=CAll['H4'],min_dist=0.001,n_neighbors=60,rstate=42)
        L1=DB1.iloc[0].Line

        L2=DB2.iloc[0].Line
        
        m=CAll['Line']==DB1.loc[0].Line
        plt.scatter(X_2d[m,0],X_2d[m,1],c='b',s=1,label=L1);
        plt.scatter(X_2d[~m,0],X_2d[~m,1],c='r',s=1,label=L2);
        plt.legend(markerscale=10)
        if fname is not None:
            plt.savefig(fname)
            
        xmax=X_2d[:,0].max()
        xmin=X_2d[:,0].min()
        ymax=X_2d[:,0].max()
        ymin=X_2d[:,0].min()
        mx=np.max([xmax,ymax])
        mn=np.min([xmin,ymin])
        
        m=CAll.Line==L1
        b=np.linspace(round(mn,0)-1,round(mx,0)+1,50)
        A,_,_=np.histogram2d(X_2d[m,0],X_2d[m,1],bins=b)
        # plt.figure(figsize=(5,5))
        # plt.imshow(A>0)
        DD=distance.cdist(X_2d[m],X_2d[m]).flatten()
        print(L1," Local: ",(A>0).sum()," Global: ",np.round(np.quantile(DD,0.95),2))

        m=CAll.Line==L2
        b=np.linspace(round(mn,0)-1,round(mx,0)+1,50)
        A,_,_=np.histogram2d(X_2d[m,0],X_2d[m,1],bins=b)
        # plt.figure(figsize=(5,5))
        # plt.imshow(A>0)
        DD=distance.cdist(X_2d[m],X_2d[m]).flatten()
        print(L2," Local: ", (A>0).sum()," Global: ",np.round(np.quantile(DD,0.95),2))
              
            
def wfall(shap_values, max_display=10, show=True):
    """ Plots an explantion of a single prediction as a waterfall plot.
    The SHAP value of a feature represents the impact of the evidence provided by that feature on the model's
    output. The waterfall plot is designed to visually display how the SHAP values (evidence) of each feature
    move the model output from our prior expectation under the background data distribution, to the final model
    prediction given the evidence of all the features. Features are sorted by the magnitude of their SHAP values
    with the smallest magnitude features grouped together at the bottom of the plot when the number of features
    in the models exceeds the max_display parameter.
    
    Parameters
    ----------
    shap_values : Explanation
        A one-dimensional Explanation object that contains the feature values and SHAP values to plot.
    max_display : str
        The maximum number of features to plot.
    show : bool
        Whether matplotlib.pyplot.show() is called before returning. Setting this to False allows the plot
        to be customized further after it has been created.
    """
    dark_o= mpl.colors.to_rgb('dimgray')
    dim_g= mpl.colors.to_rgb('darkorange')

    base_values = shap_values.base_values
    
    features = shap_values.data
    feature_names = shap_values.feature_names
    lower_bounds = getattr(shap_values, "lower_bounds", None)
    upper_bounds = getattr(shap_values, "upper_bounds", None)
    values = shap_values.values

    # make sure we only have a single output to explain
    if (type(base_values) == np.ndarray and len(base_values) > 0) or type(base_values) == list:
        raise Exception("waterfall_plot requires a scalar base_values of the model output as the first " \
                        "parameter, but you have passed an array as the first parameter! " \
                        "Try shap.waterfall_plot(explainer.base_values[0], values[0], X[0]) or " \
                        "for multi-output models try " \
                        "shap.waterfall_plot(explainer.base_values[0], values[0][0], X[0]).")

    # make sure we only have a single explanation to plot
    if len(values.shape) == 2:
        raise Exception("The waterfall_plot can currently only plot a single explanation but a matrix of explanations was passed!")
    
    # unwrap pandas series
    if safe_isinstance(features, "pandas.core.series.Series"):
        if feature_names is None:
            feature_names = list(features.index)
        features = features.values

    # fallback feature names
    if feature_names is None:
        feature_names = np.array([labels['FEATURE'] % str(i) for i in range(len(values))])
    
    # init variables we use for tracking the plot locations
    num_features = min(max_display, len(values))
    row_height = 0.5
    rng = range(num_features - 1, -1, -1)
    order = np.argsort(-np.abs(values))
    pos_lefts = []
    pos_inds = []
    pos_widths = []
    pos_low = []
    pos_high = []
    neg_lefts = []
    neg_inds = []
    neg_widths = []
    neg_low = []
    neg_high = []
    loc = base_values + values.sum()
    yticklabels = ["" for i in range(num_features + 1)]
    
    # size the plot based on how many features we are plotting
    pl.gcf().set_size_inches(8, num_features * row_height + 1.5)

    # see how many individual (vs. grouped at the end) features we are plotting
    if num_features == len(values):
        num_individual = num_features
    else:
        num_individual = num_features - 1

    # compute the locations of the individual features and plot the dashed connecting lines
    for i in range(num_individual):
        sval = values[order[i]]
        loc -= sval
        if sval >= 0:
            pos_inds.append(rng[i])
            pos_widths.append(sval)
            if lower_bounds is not None:
                pos_low.append(lower_bounds[order[i]])
                pos_high.append(upper_bounds[order[i]])
            pos_lefts.append(loc)
        else:
            neg_inds.append(rng[i])
            neg_widths.append(sval)
            if lower_bounds is not None:
                neg_low.append(lower_bounds[order[i]])
                neg_high.append(upper_bounds[order[i]])
            neg_lefts.append(loc)
        if num_individual != num_features or i + 4 < num_individual:
            pl.plot([loc, loc], [rng[i] -1 - 0.4, rng[i] + 0.4], color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
        if features is None:
            yticklabels[rng[i]] = feature_names[order[i]]
        else:
            yticklabels[rng[i]] = format_value(features[order[i]], "%0.03f") + " = " + feature_names[order[i]] 
    
    # add a last grouped feature to represent the impact of all the features we didn't show
    if num_features < len(values):
        yticklabels[0] = "%d other features" % (len(values) - num_features + 1)
        remaining_impact = base_values - loc
        if remaining_impact < 0:
            pos_inds.append(0)
            pos_widths.append(-remaining_impact)
            pos_lefts.append(loc + remaining_impact)
            c = dim_g  #colors.red_rgb
        else:
            neg_inds.append(0)
            neg_widths.append(-remaining_impact)
            neg_lefts.append(loc + remaining_impact)
            c = dark_o #colors.blue_rgb

    points = pos_lefts + list(np.array(pos_lefts) + np.array(pos_widths)) + neg_lefts + list(np.array(neg_lefts) + np.array(neg_widths))
    dataw = np.max(points) - np.min(points)
    
    # draw invisible bars just for sizing the axes
    label_padding = np.array([0.1*dataw if w < 1 else 0 for w in pos_widths])
    pl.barh(pos_inds, np.array(pos_widths) + label_padding + 0.02*dataw, left=np.array(pos_lefts) - 0.01*dataw, color=colors.red_rgb, alpha=0)
    label_padding = np.array([-0.1*dataw  if -w < 1 else 0 for w in neg_widths])
    pl.barh(neg_inds, np.array(neg_widths) + label_padding - 0.02*dataw, left=np.array(neg_lefts) + 0.01*dataw, color=colors.blue_rgb, alpha=0)
    
    # define variable we need for plotting the arrows
    head_length = 0.08
    bar_width = 0.8
    xlen = pl.xlim()[1] - pl.xlim()[0]
    fig = pl.gcf()
    ax = pl.gca()
    xticks = ax.get_xticks()
    bbox = ax.get_window_extent().transformed(fig.dpi_scale_trans.inverted())
    width, height = bbox.width, bbox.height
    bbox_to_xscale = xlen/width
    hl_scaled = bbox_to_xscale * head_length
    renderer = fig.canvas.get_renderer()
    
    # draw the positive arrows
    for i in range(len(pos_inds)):
        dist = pos_widths[i]
        arrow_obj = pl.arrow(
            pos_lefts[i], pos_inds[i], max(dist-hl_scaled, 0.000001), 0,
            head_length=min(dist, hl_scaled),
            color=dim_g, width=bar_width,
            head_width=bar_width
        )
        
        if pos_low is not None and i < len(pos_low):
            pl.errorbar(
                pos_lefts[i] + pos_widths[i], pos_inds[i], 
                xerr=np.array([[pos_widths[i] - pos_low[i]], [pos_high[i] - pos_widths[i]]]),
                ecolor=dim_g
            )

        txt_obj = pl.text(
            pos_lefts[i] + 0.5*dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                pos_lefts[i] + (5/72)*bbox_to_xscale + dist, pos_inds[i], format_value(pos_widths[i], '%+0.02f'),
                horizontalalignment='left', verticalalignment='center', color=dim_g,
                fontsize=12
            )
    
    # draw the negative arrows
    for i in range(len(neg_inds)):
        dist = neg_widths[i]
        
        arrow_obj = pl.arrow(
            neg_lefts[i], neg_inds[i], -max(-dist-hl_scaled, 0.000001), 0,
            head_length=min(-dist, hl_scaled),
            color=dark_o, width=bar_width,
            head_width=bar_width
        )

        if neg_low is not None and i < len(neg_low):
            pl.errorbar(
                neg_lefts[i] + neg_widths[i], neg_inds[i], 
                xerr=np.array([[neg_widths[i] - neg_low[i]], [neg_high[i] - neg_widths[i]]]),
                ecolor=dark_o
            )
        
        txt_obj = pl.text(
            neg_lefts[i] + 0.5*dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
            horizontalalignment='center', verticalalignment='center', color="white",
            fontsize=12
        )
        text_bbox = txt_obj.get_window_extent(renderer=renderer)
        arrow_bbox = arrow_obj.get_window_extent(renderer=renderer)
        
        # if the text overflows the arrow then draw it after the arrow
        if text_bbox.width > arrow_bbox.width: 
            txt_obj.remove()
            
            txt_obj = pl.text(
                neg_lefts[i] - (5/72)*bbox_to_xscale + dist, neg_inds[i], format_value(neg_widths[i], '%+0.02f'),
                horizontalalignment='right', verticalalignment='center', color=dark_o,
                fontsize=12
            )

    # draw the y-ticks twice, once in gray and then again with just the feature names in black
    ytick_pos = list(range(num_features)) + list(np.arange(num_features)+1e-8) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    pl.yticks(ytick_pos, yticklabels[:-1] + [l.split('=')[-1] for l in yticklabels[:-1]], fontsize=13)
    
    # put horizontal lines for each feature row
    for i in range(num_features):
        pl.axhline(i, color="#cccccc", lw=0.5, dashes=(1, 5), zorder=-1)
    
    # mark the prior expected value and the model prediction
    pl.axvline(base_values, 0, 1/num_features, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    fx = base_values + values.sum()
    pl.axvline(fx, 0, 1, color="#bbbbbb", linestyle="--", linewidth=0.5, zorder=-1)
    
    # clean up the main axis
    pl.gca().xaxis.set_ticks_position('bottom')
    pl.gca().yaxis.set_ticks_position('none')
    pl.gca().spines['right'].set_visible(False)
    pl.gca().spines['top'].set_visible(False)
    pl.gca().spines['left'].set_visible(False)
    ax.tick_params(labelsize=13)
    #pl.xlabel("\nModel output", fontsize=12)

    # draw the E[f(X)] tick mark
    xmin,xmax = ax.get_xlim()
    ax2=ax.twiny()
    ax2.set_xlim(xmin,xmax)
    ax2.set_xticks([base_values, base_values+1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax2.set_xticklabels(["\n$E[f(X)]$","\n$ = "+format_value(base_values, "%0.03f")+"$"], fontsize=12, ha="left")
    ax2.spines['right'].set_visible(False)
    ax2.spines['top'].set_visible(False)
    ax2.spines['left'].set_visible(False)

    # draw the f(x) tick mark
    ax3=ax2.twiny()
    ax3.set_xlim(xmin,xmax)
    ax3.set_xticks([base_values + values.sum(), base_values + values.sum() + 1e-8]) # The 1e-8 is so matplotlib 3.3 doesn't try and collapse the ticks
    ax3.set_xticklabels(["$f(x)$","$ = "+format_value(fx, "%0.03f")+"$"], fontsize=12, ha="left")
    tick_labels = ax3.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-10/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(12/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_color("#999999")
    ax3.spines['right'].set_visible(False)
    ax3.spines['top'].set_visible(False)
    ax3.spines['left'].set_visible(False)

    # adjust the position of the E[f(X)] = x.xx label
    tick_labels = ax2.xaxis.get_majorticklabels()
    tick_labels[0].set_transform(tick_labels[0].get_transform() + matplotlib.transforms.ScaledTranslation(-20/72., 0, fig.dpi_scale_trans))
    tick_labels[1].set_transform(tick_labels[1].get_transform() + matplotlib.transforms.ScaledTranslation(22/72., -1/72., fig.dpi_scale_trans))
    
    tick_labels[1].set_color("#999999")

    # color the y tick labels that have the feature values as gray
    # (these fall behind the black ones with just the feature name)
    tick_labels = ax.yaxis.get_majorticklabels()
    for i in range(num_features):
        tick_labels[i].set_color("#999999")
    
    if show:
        pl.show()

def dbscan_plot(data,eps=0.1,min_samples=50):
    X=data
    X = StandardScaler().fit_transform(X)
    db = DBSCAN(eps=eps, min_samples=min_samples).fit(X)
    core_samples_mask = np.zeros_like(db.labels_, dtype=bool)
    core_samples_mask[db.core_sample_indices_] = True
    labels = db.labels_

    # Number of clusters in labels, ignoring noise if present.
    n_clusters_ = len(set(labels)) - (1 if -1 in labels else 0)
    n_noise_ = list(labels).count(-1)

    print('Estimated number of clusters: %d' % n_clusters_)
    print('Estimated number of noise points: %d' % n_noise_)
    print("Silhouette Coefficient: %0.3f"
          % metrics.silhouette_score(X, labels))

    # Black removed and is used for noise instead.
    plt.figure(figsize=(10, 10))
    unique_labels = set(labels)
    colors = [plt.cm.Spectral(each)
              for each in np.linspace(0, 1, len(unique_labels))]
    for k, col in zip(unique_labels, colors):
        if k == -1:
            # Black used for noise.
            col = [0, 0, 0, 1]

        class_member_mask = (labels == k)
        
        xy = X[class_member_mask & core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),label = k,
                 markeredgecolor='k', markersize=14)
        
        xy = X[class_member_mask & ~core_samples_mask]
        plt.plot(xy[:, 0], xy[:, 1], 'o', markerfacecolor=tuple(col),
                 markeredgecolor='k', markersize=6)
    
    plt.legend(fontsize=15, title_fontsize='40')    
    plt.title('Estimated number of clusters: %d' % n_clusters_)
#    plt.show()
    return labels



def residual(params, x, data):
    alpha = params['alpha']
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H3.3']*alpha+x['H4']*beta+x['H3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H3.3'])+np.std(od['H4'])+np.std(od['H3'])


def residual2(params, x, data):
    beta = params['beta']
    gam = params['gamma']
 
 
    avMarkers=x['H4']*beta+x['H3.3']*gam
    od=x.subtract(avMarkers,axis=0)
    return np.std(od['H4'])+np.std(od['H3.3'])



def twoSampZ(X1, X2):
    from numpy import sqrt, abs, round
    from scipy.stats import norm
    mudiff=np.mean(X1)-np.mean(X2)
    sd1=np.std(X1)
    sd2=np.std(X2)
    n1=len(X1)
    n2=len(X2)
    pooledSE = sqrt(sd1**2/n1 + sd2**2/n2)
    z = ((X1 - X2) - mudiff)/pooledSE
    pval = 2*(1 - norm.cdf(abs(z)))
    return round(pval, 4)

def statistic(dframe):
    return dframe.corr().loc[Var1,Var2]


def draw_umap(data,n_neighbors=15, min_dist=0.1, n_components=2, metric='euclidean', title=''
              ,cc=0,rstate=42,dens=False):
    fit = umap.UMAP(
        n_neighbors=n_neighbors,
        min_dist=min_dist,
        n_components=n_components,
        metric=metric, random_state=rstate, verbose=True, densmap=dens
    )
    u = fit.fit_transform(data);
    plt.figure(figsize=(6, 5))
    if n_components == 2:
        plt.scatter(u[:,0], u[:,1], c=cc,s=3,cmap=plt.cm.seismic)
        plt.clim(-5,5)
        plt.colorbar()
    plt.title(title, fontsize=18)
    return u;


def NormMark(data):
    params = Parameters()
    params.add('beta', value=0.1, min=0)
    params.add('gamma', value=0.1, min=0)
    params.add('alpha', value=0.1, min=0)
    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value
    alpha=out.params['alpha'].value
    avMarkers=ddf['H3.3']*alpha+ddf['H4']*beta+ddf['H3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols]=data[EpiCols]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data

def NormMark2(data):
    params = Parameters()
    params.add('beta', value=0.1, min=-1000)
    params.add('gamma', value=0.1, min=-1000)

    ddf=data.copy()
    ddf2=data.copy()
    out = minimize(residual2, params, args=(ddf, ddf),method='cg')
    beta=out.params['beta'].value
    gam=out.params['gamma'].value

    avMarkers=ddf['H4']*beta+ddf['H3.3']*gam
    ddf=ddf.subtract(avMarkers,axis=0)
    data=ddf
    ddf2[EpiCols_M]=data[EpiCols_M]
#    BCKData[NamesAll]=data[NamesAll]
    data=ddf2.copy()
    del ddf
    del ddf2
    return data






def f(): raise Exception("Found exit()")



def BPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.boxplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   

def VPlots(data,NMS,xVar='type'):
    for NN in NMS:
        BoxVar=NN
        plt.figure(figsize=(3, 5))    
        ax = sns.violinplot(x=xVar, y=NN, data=data,showfliers=False,palette=['red','blue'])
        plt.title(NN+" MGG")
        plt.show()   


def KPlots(data,NMS,titleSup=''):
    for NN in NMS:
        plt.figure(figsize=(10,10))
        sns.kdeplot(data=data,x=NN,color='blue')
        
#        plt.legend()
        plt.title(""+NN+" "+titleSup)
        plt.show()



def MeanDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

    
def MedDist(data1,data2,Markers,title='',clr=['darkgreen','purple']):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].median().sort_values(ascending=False)
    dd1=data2[Markers].median().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    

    colors = [clr[0] if x < 0 else clr[1] for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)    
    
def MeanDistIdU(data1,data2,Markers,title=''):
    sns.set_style({'legend.frameon':True})
 
    dd0=data1[Markers].mean().sort_values(ascending=False)
    dd1=data2[Markers].mean().sort_values()
    diffs=(dd1-dd0).sort_values(ascending=False)    
    colors = ['dodgerblue' if x < 0 else 'darkmagenta' for x in diffs]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs, color=colors, alpha=1, linewidth=5)
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)

def KPlot_Mrk(Mark,titleSup=''):
    plt.figure(figsize=(10,10))
    sns.kdeplot(data=C01,x=Mark,label="C01")
    sns.kdeplot(data=C02,x=Mark,label="C02")
    sns.kdeplot(data=C03,x=Mark,label="C03")
    sns.kdeplot(data=C04,x=Mark,label="C04")
    sns.kdeplot(data=C05,x=Mark,label="C05")
    plt.legend()
    plt.title(""+Mark+" "+titleSup)
    plt.show()
    
    
def MeanDistReSamp(data1,data2,Markers,title='',clr=['darkgreen','purple'],nsamp=10,f=0.5):
    sns.set_style({'legend.frameon':True})
    diffs=[]
    for i in range(nsamp):  
        D1=data1.sample(frac=f).copy()
        D2=data2.sample(frac=f).copy()
        dd0=D1[Markers].mean()#.sort_values(ascending=False)
        dd1=D2[Markers].mean()#.sort_values()
        diff=(dd1-dd0)#.sort_values(ascending=False)    
        diffs.append(diff)

    Mdiff=np.asarray(diffs)
    D=pd.DataFrame({'M':Mdiff.mean(axis=0),'S':Mdiff.std(axis=0)},index=Markers)    
    
    diffs=D.sort_values(by='M',ascending=False).copy()
    
    
    colors = [clr[0] if x < 0 else clr[1] for x in diffs.M]
    
    fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
    plt.hlines(y=diffs.index, xmin=0, xmax=diffs.M, color=colors, alpha=1, linewidth=5)
    plt.errorbar(y=diffs.index,x=diffs.M,xerr=diffs.S,capsize=5,fmt='k.')
    # Decorations
    plt.gca().set(ylabel='', xlabel='')
    plt.xticks(fontsize=20 ) 
    plt.yticks(fontsize=16 ) 

    plt.title(title, fontdict={'size':20})
    plt.grid(linestyle='--', alpha=0.5)            
    

def UMAP_Plot(data1,data2,Markers,Set1='C01',Set2='Other',titleSup=''):
    data1=data1.assign(Set=Set1)
    data2=data2.assign(Set=Set2)
    CAll=data1.append(data2).sample(frac=0.1).copy()
    print(CAll)
    X_2d=draw_umap(CAll[Markers],cc=CAll['H3'],min_dist=0.01)
    for NN in NamesAll:
        cc=CAll[NN]#[mask]
        plt.figure(figsize=(6, 5))
        plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                    c=cc, cmap=plt.cm.jet)
    #    cmap = matplotlib.cm.get_cmap('jet')
        plt.colorbar()
    #    plt.clim(-3.5,3.5)
        plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    #    mask=CAllmask[TSNEVar]==True
    #    rgba = cmap(-10)
    #    plt.scatter(X_2d[mask][:,0],X_2d[mask][:,1],s=2,
    #                color=rgba) 
        plt.title(NN+" "+titleSup)
        plt.show()

    plt.figure(figsize=(6, 5))
    mask=CAll.Set==Set1
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='blue', label=Set1)        
    mask=CAll.Set==Set2
    plt.scatter(X_2d[mask,0],X_2d[mask,1],s=2,
            c='red', label=Set2)        
    plt.legend()
    plt.show()
       

def DeltaCorr(data1,data2,Markers,titleSup=''):
    params = {'axes.titlesize': 30,
              'legend.fontsize': 20,
              'figure.figsize': (16, 10),
              'axes.labelsize': 20,
              'axes.titlesize': 20,
              'xtick.labelsize': 16,
              'ytick.labelsize': 16,
              'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

    print(titleSup)
    plt.figure(figsize=(20,20))
    matrix=data2[Markers].corr()-data1[Markers].corr()
    g=sns.clustermap(matrix, annot=True, annot_kws={"size":8},
                     cmap=plt.cm.jet,vmin=matrix.min().min(),vmax=matrix.max().max(),linewidths=.1); 
    plt.xticks(rotation=0); 
    plt.yticks(rotation=0); 

    plt.title(titleSup)
    plt.show()
    
    
def DefStyle():
    params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
    plt.rcParams.update(params)
    plt.style.use('seaborn-whitegrid')
    sns.set_style("white")

# Load and initialize

In [ ]:
NamesAll=['H3',
'Cytokeratin5',
'H3K27me2',
'p53',
'EZH2',
'H3K4me3',
'H3K36me2',
'H3K4me1',
'H3K9me2',
'H4K16ac',
'H2AK119Ub',
'H3.3',
'H3K64ac',
'ZEB1',
'H4',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'human-EpCAM',
'yH2A.X',
'Vimentin',
'ER',
'CD49f',
'CD24',
'GATA3',
'CD44',
'Ki-67',
'K8_18']


IdentityCols=[
'Cytokeratin5',
'GATA3',
'human-EpCAM',
'Vimentin',
'ER',
'CD49f',
'CD24',
'CD44',
'K8_18',
'ZEB1']

innerCols = ['H3',
'Ki-67',
'p53',
'EZH2',
'Cytokeratin5',
'GATA3',
'Vimentin',
'ER',
'K8_18',
'ZEB1',
'H3K27me2',
'H3K4me3',
'H3K36me2',
'H3K4me1',
'H3K9me2',
'H4K16ac',
'H2AK119Ub',
'H3.3',
'H3K64ac',
'H4',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'yH2A.X',
]


EpiCols=['H3',
'H3K27me2',
'H3K4me3',
'H3K36me2',
'H3K4me1',
'H3K9me2',
'H4K16ac',
'H2AK119Ub',
'H3.3',
'H3K64ac',
'H4',
'H3K27ac',
'H4K20me3',
'H3K36me3',
'H3K27me3',
'H3K9ac',
'H3K9me3',
'H3S28p',
'yH2A.X',
]

metals_names_map = {' In115Di':'H3', ' Ce140Di':'Cytokeratin5', ' Nd142Di':'H3K27me2', ' Nd143Di':'p53', ' Nd144Di':'EZH2', ' Nd145Di':'H3K4me3',
                   ' Sm149Di':'H3K36me2', ' Nd150Di':'H3K4me1', ' Eu151Di':'H3K9me2', ' Sm152Di':'H4K16ac', ' Eu153Di':'H2AK119Ub', ' Gd155Di':'H3.3',
                   ' Gd156Di':'H3K64ac', ' Gd158Di':'ZEB1', ' Tb159Di':'H4', ' Gd160Di':'H3K27ac', ' Dy161Di':'H4K20me3', ' Ho165Di':'H3K36me3',
                   ' Er168Di':'H3K27me3', ' Tm169Di':'H3K9ac', ' Er170Di':'H3K9me3', ' Lu175Di':'H3S28p', ' Pr141Di':'human-EpCAM',
                    ' Sm147Di':'yH2A.X', ' Sm154Di':'Vimentin', ' Dy163Di':'ER', ' Dy164Di':'CD49f', ' Er166Di':'CD24', ' Er167Di':'GATA3',
                   ' Yb171Di':'CD44', ' Yb172Di':'Ki-67', ' Yb174Di':'K8_18'}


NamesAll_M=['H3.3', 'H4', 'H3K27Ac', 'H3K27me3', 'Irridium']
EpiCols_M=['H3.3', 'H4', 'H3K27Ac', 'H3K27me3']

In [ ]:
dir="../csv_files/"
MDAMB468_w_CO2=pd.read_csv(dir+"c18_export_CyTOF_BClines_christi_06Feb2023_01_1_0_Time, Width subset_MCF7_Yael.csv")

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")

MDAMB468_w_CO2.rename(columns = metals_names_map, inplace=True)




MDAMB468_w_CO2=MDAMB468_w_CO2[NamesAll]


In [ ]:
sns.kdeplot(data=MDAMB468_w_CO2,x='Ki-67',color='green',zorder=10)


# Gate on H3.3/H4 too low, but also remove outliers 99.99% from all 

In [ ]:
GateColumns=['H3.3','H4','H3']#,'H3']#,'H3']



def Gate(data,name):
    ddf=data.copy()
    print(name)
    print("Initial ",len(ddf))
    ddf=ddf[(ddf[GateColumns]>5).all(axis=1)]
    print("Core Gate ",len(ddf))
    ddf=ddf[(ddf<np.quantile(ddf,0.9999,axis=0)).all(axis=1)]
    print("Outlier Gate ",len(ddf))
    data=ddf.copy()
    del ddf
    return data

MDAMB468_w_CO2=Gate(MDAMB468_w_CO2,"MDAMB468_w_CO2")

In [ ]:
BCK=MDAMB468_w_CO2.copy()

In [ ]:

scFac=5
MDAMB468_w_CO2=np.arcsinh(MDAMB468_w_CO2/scFac)


In [ ]:
sns.kdeplot(MDAMB468_w_CO2['Ki-67'],color='green',label='MDAMB468_w_CO2')
plt.legend(bbox_to_anchor=(1.,1.),loc='upper left')
#plt.savefig('Plots/CyTOF3_Ki67.pdf',dpi=200,bbox_inches='tight')

# Normalize using new method on all intercellular markers

In [ ]:
def R(p,x,data,Q,M,M1,M2,M3):
    a=p['a']
    b=p['b']
    d=x.divide(a*M1+(1-a-b)*M2+b*M3,axis=0)
    return d.std()['H3.3']+d.std()['H4']+d.std()['H3']

def NormalizeNew(data):
    
    params = Parameters()
    params.add('a', value=0.1,min=0,max=1)
    params.add('b', value=0.1,min=0,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4','H3']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
    M3=(ddf/Q)['H3']

    out=minimize(R, params ,args=(ddf, ddf,Q,M,M1,M2,M3),method='cg')
    AA=out.params['a'].value
    BB=out.params['b'].value
    M=M1*AA+M2*(1-AA-BB)+M3*BB
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf
    ddf2[innerCols]=data[innerCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data
    
def R2(p,x,data,Q,M,M1,M2):
    a=p['a']
    d=x.divide(a*M1+(1-a)*M2,axis=0)
    return (d.std()['H3.3'])**2+(d.std()['H4'])**2

def NormalizeNew2(data):
    
    params = Parameters()
    params.add('a', value=0.5,min=0.3,max=1)
    ddf=data.copy()
    ddf2=data.copy()
    Q=ddf.mean()
    M=(ddf/Q)[['H3.3','H4']].mean(axis=1)
    M1=(ddf/Q)['H3.3']
    M2=(ddf/Q)['H4']
 
    out=minimize(R2, params ,args=(ddf, ddf,Q,M,M1,M2),method='cg')
    AA=out.params['a'].value

    M=M1*AA+M2*(1-AA)
    ddf=ddf.divide(M,axis=0).copy()
    data=ddf.copy()
    print(data.shape,ddf2.shape)
    ddf2[innerCols]=data[innerCols]
    data=ddf2.copy()
    del ddf 
    del ddf2
    return data

In [ ]:

print("MDAMB468_w_CO2")
print(MDAMB468_w_CO2.std()['H3.3']+MDAMB468_w_CO2.std()['H4'])#+C01.std()['H3'])
MDAMB468_w_CO2=NormalizeNew(MDAMB468_w_CO2)
print(MDAMB468_w_CO2.std()['H3.3']+MDAMB468_w_CO2 .std()['H4'])#+C01.std()['H3'])


In [ ]:
from tqdm import tqdm
Mean_Core=MDAMB468_w_CO2[['H3.3','H4', 'H3']].mean(axis=1)
for N in tqdm(innerCols):
    MDAMB468_w_CO2[N]=MDAMB468_w_CO2[N]/Mean_Core



In [ ]:
aaa = MDAMB468_w_CO2.copy()

m=np.mean(aaa,axis=0)
s=np.std(aaa,axis=0)

MDAMB468_w_CO2=(MDAMB468_w_CO2-m)/s

params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 20,
          'ytick.labelsize': 20,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")

In [ ]:
MDAMB468_w_CO2['Line']='MCF7'


# UMAPS

In [ ]:
EPC=EpiCols.copy()
EPC.remove('H3')
EPC.remove('H3.3')
EPC.remove('H4')
CNum=60000
idx=np.random.choice(len(MDAMB468_w_CO2),replace=False,size=CNum)

In [ ]:
CAll=MDAMB468_w_CO2.iloc[idx].copy()

In [ ]:
#CAll= pd.concat([ MDAMB468_w_CO2.sample(n=CNum,random_state=42)]).copy()
X_2d=draw_umap(CAll[EPC],cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()

In [ ]:
for NN in ['Cytokeratin5','GATA3']:
#    NN='Cytokeratin5'
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()
    
#    plt.clim(-3,2)
    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title(TSNEVar)

    plt.show()


In [ ]:
labels=dbscan_plot(X_2d,eps=0.11,min_samples=90)
#plt.savefig('Plots/20230523/MDAMB468_w_CO2_dbscan_clustered.png',dpi=200,bbox_inches='tight')

In [ ]:
CAll['Cl']=labels

In [ ]:
prop_cycle = plt.rcParams['axes.prop_cycle']
colors = prop_cycle.by_key()['color']


In [ ]:
colors=[
 '#1f77b4',
 
 '#2ca02c','#ff7f0e',
 '#d62728',
 '#9467bd',
 '#8c564b',
 '#e377c2',
 '#7f7f7f',
 '#bcbd22',
 '#17becf']



In [ ]:
plt.figure(figsize=(5,5))
for i in range(5):
    m=CAll.Cl==i
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,
                label='Cluster '+str(i),c=colors[i])

plt.legend(markerscale=10,bbox_to_anchor=(1,1))
plt.savefig('Plots/20230905/MCF7/MCF7_Clusters.png',dpi=200,bbox_inches='tight')

In [ ]:
for NN in NamesAll:
#    NN='Cytokeratin5'
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()
    
#    plt.clim(-3,2)
    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title(TSNEVar)
    plt.savefig('Plots/20230905/MCF7/MCF7_'+NN+'.png',dpi=200,bbox_inches='tight')
    plt.show()


In [ ]:
EPC=[
 'Cytokeratin5',
 'H3K27me2',
 'p53',
 'EZH2',
 'H3K4me3',
 'H3K36me2',
 'H3K4me1',
 'H3K9me2',
 'H4K16ac',
 'H2AK119Ub',
 'H3K64ac',
 'ZEB1',
 'H3K27ac',
 'H4K20me3',
 'H3K36me3',
 'H3K27me3',
 'H3K9ac',
 'H3K9me3',
 'H3S28p',
 'human-EpCAM',
 'yH2A.X',
 'Vimentin',
 'ER',
 'CD49f',
 'CD24',
 'GATA3',
 'CD44',
 'Ki-67',
 'K8_18']
EPC.sort()

In [ ]:
Mat=CAll[CAll.Cl>-1].groupby('Cl').mean(numeric_only=True)

In [ ]:
plt.figure(figsize=(5,12))
sns.clustermap(Mat[EPC].T,cmap=plt.cm.seismic,annot=True,yticklabels=True,)
plt.savefig('Plots/20230905/MCF7/MCF7HeatMap.png',dpi=200,bbox_inches='tight')

In [ ]:
BCK['Cl']=-1

In [ ]:
B_Subset=BCK.iloc[idx].copy()

In [ ]:
B_Subset['Cl']=labels

In [ ]:
sns.scatterplot(data=BCK,y='Cytokeratin5',x='GATA3',s=.5)
plt.yscale('log')
plt.xscale('log')

In [ ]:
B_Subset[IdentityCols]

In [ ]:
pp.axes.shape

In [ ]:
NMS=['GATA3','Cytokeratin5','']
pp=sns.pairplot(data=B_Subset[IdentityCols].sample(10000),corner=False,plot_kws={"s":1})
for ax in pp.axes.flat:
        ax.set_xscale('log')
        ax.set_yscale('log')


In [ ]:
LN=len(IdentityCols)

In [ ]:
LN

In [ ]:
dir

In [ ]:
Colors=['r','orange','yellow','green','blue']

for j in range(LN):
    for k in range(j,LN):

        plt.figure()
        for i,Cl in enumerate([0,1,2,3,4]):
            m=B_Subset.Cl==Cl
            sns.scatterplot(data=B_Subset[m],y=IdentityCols[j],x=IdentityCols[k],s=.5,label=Cl,color=Colors[i])
            plt.legend(loc="upper right",bbox_to_anchor=(1.5,1),markerscale=10)
            plt.yscale('log')
            plt.xscale('log')
            plt.xlim(1,5000)
            plt.xlim(1,1e4)


        plt.legend(loc="upper right",bbox_to_anchor=(1.5,1),markerscale=10)
        plt.yscale('log')
        plt.xscale('log')
        plt.savefig("/Users/ronguy/Dropbox/CyTOF_Breast/CyTOF_CR7/Plots/20230523/2D_Projected_On_Raw_"+IdentityCols[j]+"-"+IdentityCols[k]+".png", bbox_inches='tight')

In [ ]:
B_Subset.groupby('Cl').mean()

In [ ]:
sns.kdeplot(data=CAll.loc[CAll.Cl==0,:],x='Cytokeratin5',color='green',zorder=10)
sns.kdeplot(data=CAll.loc[CAll.Cl==1,:],x='Cytokeratin5',color='red',zorder=10)


In [ ]:
CAll["Cl"]=labels

In [ ]:
CAll.to_csv("match_labels.csv")

In [ ]:
CAll['Cl']=labels
CAll['LineB']=CAll.Cl
m = CAll.Cl == 1
CAll.loc[m,'LineB']=2
m = CAll.Cl == 2
CAll.loc[m,'LineB']=1

CAll.Cl=CAll.LineB

In [ ]:
def drawHeatmap(df, title, subtitle='', xlabel=''):
    extremes = np.quantile(df.values,0.99), np.quantile(df.values,0.01)
    df = df.copy().round(4)
    #sns.set(font_scale = 1.1)
    #fig, ax = plt.subplots(figsize=(15, 15))
    sns.clustermap(df, cmap='jet', square=True, linewidths=0.5, 
        fmt = '',cbar_kws={"shrink": 0.6}, 
        annot_kws={'size': 'small', 'alpha': 0.9}, annot=True, 
        vmax=extremes[0], vmin=extremes[1], xticklabels=xlabel)
    #plt.=xlabel
    plt.text(x=5.2, y=2, s=title, fontsize=24, weight='bold')
    plt.text(x=0, y=1.5, s=subtitle, fontsize=18, alpha=0.75)

    #plt.title(title, fontsize = 30)



In [ ]:
params = {'axes.titlesize': 30,
          'legend.fontsize': 20,
          'figure.figsize': (6, 5),
          'axes.labelsize': 20,
          'axes.titlesize': 20,
          'xtick.labelsize': 15,
          'ytick.labelsize': 12,
          'figure.titlesize': 30}
plt.rcParams.update(params)
plt.style.use('seaborn-whitegrid')
sns.set_style("white")


m=(CAll.Cl!=-1)
CAll = CAll.loc[m,:]
CAll["count"]=np.ones(CAll.shape[0])
sums = CAll.groupby("Cl").sum()

NamesPresent = NamesAll.copy()
NamesPresent.remove("H3")
NamesPresent.remove("H3.3")
NamesPresent.remove("H4")
means = CAll.groupby("Cl")[NamesPresent].mean()

xlabel=[]
sums['line'] = sums.index
sum_all=sum(sums["count"].values)
for index, row in sums.iterrows():
    xlabel.append(str(row["line"])[:-2] +": "+str(round(row['count']))+'\n'+ str(round(row['count']*100/sum_all,2))+"%")
print(xlabel)
drawHeatmap(means.T, "Means Per Cluster - Based On HM, "+CAll.Line[1], xlabel=xlabel)
plt.savefig("Plots/heatmap_per_cluster.png", bbox_inches='tight')

In [ ]:
areas = {}
het = {}
for L in CAll.Cl.unique():
    m= CAll.Cl==L
    rx = (np.max(X_2d[m,0])-np.min(X_2d[m,0]))/2
    ry = (np.max(X_2d[m,1])-np.min(X_2d[m,1]))/2
    areas[L] = np.pi*rx*ry
    het[L] = areas[L]/sum(m)
print(areas)
het.pop(-1)
het.pop(1)
print(het)
print(min(het.values()))

In [ ]:
CAll['LineB']=CAll['Line']
CAll['Cl']=labels
# m=CAllB.LineB=='OCI-Ly7'
# plt.scatter(X_2d[m,0],X_2d[m,1],s=1,c='red',label='OCI-Ly7')
# m=CAllB.LineB=='EZH2 Y646N'
# plt.scatter(X_2d[m,0],X_2d[m,1],s=1,c='blue',label='EZH2 Y646N')
# m=CAllB.LineB=='EZH2 Y646N WT-Like'
# plt.scatter(X_2d[m,0],X_2d[m,1],s=1,c='plum',label='EZH2 Y646N WT-Like')
# plt.legend(markerscale=10,bbox_to_anchor=(1,1))
# #plt.savefig('Plots/UMAP_Ly7_EZH2_NoG0.pdf',dpi=200,bbox_inches='tight')

In [ ]:
Names=NamesAll.copy()
Names.remove('H3')
Names.remove('H3.3')
Names.remove('H4')
# Names.remove('H3K64ac')
# Names.remove('H4K16ac')

sns.set_style({'legend.frameon':True})

dd0=np.mean(CAll[CAll.Cl==0][Names],axis=0).sort_values(ascending=False)
dd1=np.mean(CAll[CAll.Cl==1][Names],axis=0).sort_values()
dd2=np.mean(CAll[CAll.Cl==2][Names],axis=0).sort_values()
dd3=np.mean(CAll[CAll.Cl==3][Names],axis=0).sort_values()



sz0=np.full(len(dd0),0,dtype=np.float64)
sz1=np.full(len(dd1),0,dtype=np.float64)
sz2=np.full(len(dd2),0,dtype=np.float64)
sz3=np.full(len(dd3),0,dtype=np.float64)

    
fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
ax.hlines(y=dd0.index, xmin=-5, xmax=5, color='gray', alpha=0.7, 
          linewidth=1, linestyles='dashdot')

ax.scatter(y=dd0.index, x=dd0, s=100, c='red', alpha=1,
           label='clust 0',)
ax.scatter(y=dd1.index, x=dd1, s=100, c='orange', alpha=1,
           label='clust 1',)
ax.scatter(y=dd2.index, x=dd2, s=100, c='yellow', alpha=1,
           label='clust 2',)
ax.scatter(y=dd3.index, x=dd3, s=100, c='green', alpha=1,
           label='clust 3',)
#ax.scatter(y=dd4.index, x=dd2, s=100, c='cyan', alpha=1,label='clust 4',)



ax.vlines(x=0, ymin=0, ymax=len(dd0)-1, color='black', alpha=0.7, linewidth=2, linestyles='dotted')
plt.legend(fontsize=30,
           facecolor='White', framealpha=1,frameon=True,
           bbox_to_anchor=(1.0, .50, 0.3, 0.2), loc='upper left')

ax.set_title('Mean Value - '+CAll.Line[1], fontdict={'size':30})
#ax.set_xlim(-1.5, 1.5)

labels = dd0.index.to_list()
#labels[8]="H3-K27M"
ax.set_yticklabels(labels)
ax.set_xlim([-3,3])
plt.setp(ax.get_xticklabels(), fontsize=24)
plt.setp(ax.get_yticklabels(), fontsize=20)

plt.savefig('Plots/Pride_HM_by_clusters.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
Names=EpiCols.copy()
Names.remove('H3')
Names.remove('H3.3')
Names.remove('H4')
# Names.remove('H3K64ac')
# Names.remove('H4K16ac')

sns.set_style({'legend.frameon':True})

dd0=np.var(CAll[CAll.Cl==0][EpiCols],axis=0).sort_values(ascending=False)
dd1=np.var(CAll[CAll.Cl==1][EpiCols],axis=0).sort_values()
dd2=np.var(CAll[CAll.Cl==2][EpiCols],axis=0).sort_values()
dd3=np.var(CAll[CAll.Cl==3][EpiCols],axis=0).sort_values()
dd4=np.var(CAll[CAll.Cl==4][EpiCols],axis=0).sort_values()



sz0=np.full(len(dd0),0,dtype=np.float64)
sz1=np.full(len(dd1),0,dtype=np.float64)
sz2=np.full(len(dd2),0,dtype=np.float64)
sz3=np.full(len(dd3),0,dtype=np.float64)
sz4=np.full(len(dd4),0,dtype=np.float64)

    
fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
ax.hlines(y=dd0.index, xmin=-5, xmax=5, color='gray', alpha=0.7, 
          linewidth=1, linestyles='dashdot')

ax.scatter(y=dd0.index, x=dd0, s=100, c='red', alpha=1,
           label='clust 0',)
ax.scatter(y=dd1.index, x=dd1, s=100, c='orange', alpha=1,
           label='clust 1',)
ax.scatter(y=dd2.index, x=dd2, s=100, c='yellow', alpha=1,
           label='clust 2',)
ax.scatter(y=dd3.index, x=dd2, s=100, c='green', alpha=1,
           label='clust 3',)
#ax.scatter(y=dd4.index, x=dd2, s=100, c='cyan', alpha=1,label='clust 4',)



ax.vlines(x=0, ymin=0, ymax=len(dd0)-1, color='black', alpha=0.7, linewidth=2, linestyles='dotted')
plt.legend(fontsize=30,
           facecolor='White', framealpha=1,frameon=True,
           bbox_to_anchor=(1.0, .50, 0.3, 0.2), loc='upper left')

ax.set_title('Variance', fontdict={'size':30})
#ax.set_xlim(-1.5, 1.5)

labels = dd0.index.to_list()
#labels[8]="H3-K27M"
ax.set_yticklabels(labels)
ax.set_xlim([-2.5,2.5])
plt.setp(ax.get_xticklabels(), fontsize=24)
plt.setp(ax.get_yticklabels(), fontsize=24)

#plt.savefig('Plots/Pride_HM_by_clusters.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
Names=EpiCols.copy()
Names.remove('H3')
Names.remove('H3.3')
Names.remove('H4')
# Names.remove('H3K64ac')
# Names.remove('H4K16ac')

sns.set_style({'legend.frameon':True})

dd0=np.mean(CAll[CAll.Cl==0][IdentityCols],axis=0).sort_values(ascending=False)
dd1=np.mean(CAll[CAll.Cl==1][IdentityCols],axis=0).sort_values()
dd2=np.mean(CAll[CAll.Cl==2][IdentityCols],axis=0).sort_values()
dd3=np.mean(CAll[CAll.Cl==3][IdentityCols],axis=0).sort_values()



sz0=np.full(len(dd0),0,dtype=np.float64)
sz1=np.full(len(dd1),0,dtype=np.float64)
sz2=np.full(len(dd2),0,dtype=np.float64)
sz3=np.full(len(dd3),0,dtype=np.float64)

    
fig, ax = plt.subplots(figsize=(16,10), dpi= 80)
ax.hlines(y=dd0.index, xmin=-5, xmax=5, color='gray', alpha=0.7, 
          linewidth=1, linestyles='dashdot')

ax.scatter(y=dd0.index, x=dd0, s=100, c='red', alpha=1,
           label='clust 0',)
ax.scatter(y=dd1.index, x=dd1, s=100, c='orange', alpha=1,
           label='clust 1',)
ax.scatter(y=dd2.index, x=dd2, s=100, c='yellow', alpha=1,
           label='clust 2',)
ax.scatter(y=dd3.index, x=dd2, s=100, c='green', alpha=1,
           label='clust 3',)



ax.vlines(x=0, ymin=0, ymax=len(dd0)-1, color='black', alpha=0.7, linewidth=2, linestyles='dotted')
plt.legend(fontsize=30,
           facecolor='White', framealpha=1,frameon=True,
           bbox_to_anchor=(1.0, .50, 0.3, 0.2), loc='upper left')

ax.set_title('Mean Value', fontdict={'size':30})
#ax.set_xlim(-1.5, 1.5)

labels = dd0.index.to_list()
#labels[8]="H3-K27M"
ax.set_yticklabels(labels)
ax.set_xlim([-2.5,2.5])
plt.setp(ax.get_xticklabels(), fontsize=24)
plt.setp(ax.get_yticklabels(), fontsize=24)

plt.savefig('Plots/Pride_identity_by_dbscan_clusters.png',dpi=200,bbox_inches='tight')
plt.show()

In [ ]:
CAll['LineB']=CAll['Line']
m=X_2d[:,1]>7
print(sum(m)/len(m))
plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
#m=(CAll['Line']=='EZH2 Y646N') & (((CAll['Cl']==0) & (X_2d[:,1]<7.12)) | (CAll.Cl>1))
#CAll.loc[m,'LineB']='EZH2 Y646N WT-Like'
MeanDistReSamp(CAll.iloc[m,:],CAll.iloc[~m,:],NamesAll,title='left lower cluster - all',clr=['red','blue'],nsamp=100,f=0.5)
#plt.savefig('Plots/MeanDistReSamp_lower_cluster_vs_all.png',dpi=200,bbox_inches='tight')

In [ ]:
CAll['LineB']=CAll['Line']
m=(X_2d[:,0]>13.5) & (X_2d[:,1]>6.5)
plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
#m=(CAll['Line']=='EZH2 Y646N') & (((CAll['Cl']==0) & (X_2d[:,1]<7.12)) | (CAll.Cl>1))
#CAll.loc[m,'LineB']='EZH2 Y646N WT-Like'
MeanDistReSamp(CAll.iloc[m,:],CAll.iloc[~m,:],NamesAll,title='right upper cluster - all',clr=['red','blue'],nsamp=100,f=0.5)
plt.savefig('Plots/MeanDistReSamp_right_upper_cluster_vs_all.png',dpi=200,bbox_inches='tight')

# UMAP by all cols

In [ ]:
CAll=pd.concat([HCC70.sample(n=CNum,random_state=42),
                HCC1937.sample(n=CNum,random_state=42),
                SUM149.sample(n=CNum,random_state=42),
                MCF7_Ori.sample(n=CNum,random_state=42),
                MCF7_Yael.sample(n=CNum,random_state=42),
                MDAMB468.sample(n=CNum,random_state=42),
                MDAMB468_wo_CO2.sample(n=CNum,random_state=42)]).copy()
X_2d=draw_umap(CAll[NamesAll],cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()

In [ ]:
Colors={}
Colors['HCC70']='blue'
Colors['HCC1937']='red'
Colors['SUM149']='cyan'
Colors['MCF7_Ori']='pink'
Colors['MCF7_Yael']='green'
Colors['MDAMB468']='orange'
Colors['MDAMB468_wo_CO2']='purple'

#'#d0384e', '#ee6445', '#fa9b58', '#fece7c', '#fff1a8', 
#'#f4faad', '#d1ed9c', '#97d5a4', '#5cb7aa', '#3682ba'

plt.figure()
for L in CAll.Line.unique():
    m=CAll.Line==L
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
    
plt.legend(bbox_to_anchor=(1,1),markerscale=10)
plt.savefig('Plots/CyTOF1_All_by_All.png',dpi=200,bbox_inches='tight')

In [ ]:
    L = 'MCF7_Yael'
    m=CAll.Line==L
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)    
    L = 'MCF7_Ori'
    m=CAll.Line==L
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)

In [ ]:
CAll['LineB']=CAll['Line']
m=(X_2d[:,0]>4) & (X_2d[:,1]>4.7)
plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
#m=(CAll['Line']=='EZH2 Y646N') & (((CAll['Cl']==0) & (X_2d[:,1]<7.12)) | (CAll.Cl>1))
#CAll.loc[m,'LineB']='EZH2 Y646N WT-Like'
MeanDistReSamp(CAll.iloc[m,:],CAll.iloc[~m,:],NamesAll,title='right upper cluster - all',clr=['red','blue'],nsamp=100,f=0.5)
plt.savefig('Plots/MeanDistReSamp_right_upper_cluster_vs_all_AllColsClust.png',dpi=200,bbox_inches='tight')

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title(TSNEVar)
    plt.savefig('Plots/All_by_All_'+NN+'_CyTOF1.png',dpi=200,bbox_inches='tight')

    plt.show()

# UMAP by identity cols

In [ ]:
EPC=EpiCols.copy()
EPC.remove('H3')
EPC.remove('H3.3')
EPC.remove('H4')
CNum=60000
CAll= pd.concat([
                MDAMB468_wo_CO2.sample(n=CNum,random_state=42)]).copy()
X_2d=draw_umap(CAll[IdentityCols],cc=CAll['H4'],min_dist=0.001,n_neighbors=40,rstate=42)
plt.show()

In [ ]:
Colors={}
Colors['MDAMB468']='red'
Colors['MDAMB468_wo_CO2']='blue'
plt.figure()
for L in CAll.Line.unique():
    m=CAll.Line==L
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
    plt.legend(bbox_to_anchor=(1,1),markerscale=10)
    plt.show()
#plt.savefig('Plots/MDAMB468_w_wo_CO2.png',dpi=200,bbox_inches='tight')

In [ ]:
labels=dbscan_plot(X_2d,eps=0.1,min_samples=90)
#plt.savefig('Plots/MDAMB468_dbscan_clustered.png',dpi=200,bbox_inches='tight')

In [ ]:
CAll['LineB']=CAll['Line']
CAll['Cl']=labels

In [ ]:
Colors={}
Colors['MDAMB468']='red'
Colors['MDAMB468_wo_CO2']='blue'
plt.figure()
for L in CAll.Line.unique():
    m=(CAll.Line==L) & (CAll.Cl==2)
    print(sum(m))
    plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
    plt.legend(bbox_to_anchor=(1,1),markerscale=10)
    plt.show()


In [ ]:
m=CAll.Cl==2
plt.scatter(X_2d[m,0],X_2d[m,1],s=1,color=Colors[L],label=L)
#m=(CAll['Line']=='EZH2 Y646N') & (((CAll['Cl']==0) & (X_2d[:,1]<7.12)) | (CAll.Cl>1))
#CAll.loc[m,'LineB']='EZH2 Y646N WT-Like'
MeanDistReSamp(CAll.loc[m,:],CAll.loc[~m,:],NamesAll,title='little tail - all',clr=['red','blue'],nsamp=100,f=0.5)
#plt.savefig('Plots/MeanDistReSamp_right_upper_cluster_vs_all.png',dpi=200,bbox_inches='tight')

In [ ]:
for NN in NamesAll:
    Var=NN
    TSNEVar=NN
    cc=CAll[NN]#[mask]
    plt.figure(figsize=(6, 5))
    plt.scatter(X_2d[:,0],X_2d[:,1],s=2,
                c=cc, cmap=plt.cm.seismic)

    plt.colorbar()

    plt.clim(cc.quantile(0.01),cc.quantile(0.99))
    plt.title(TSNEVar)
    #plt.savefig('Plots/All_by_Identity_'+NN+'_CyTOF1.png',dpi=200,bbox_inches='tight')

    plt.show()